# Multi-market 1-minute futures download

Downloads **CL, GC, SI, HG, NG, NQ** (continuous front month, real prices, all sessions) from 2022 to today and saves one `.csv.gz` per market into this project folder.

1. Run the code cell (takes a few minutes).
2. Refresh the Explorer (or reopen the project).
3. Right-click each `XX_1m_2022-2026.csv.gz` → **Download**, then upload them to Claude.

If one market hits a memory limit, shorten `MARKETS` or `YEARS` and run again.

In [ ]:
# ===== Multi-market 1-minute futures download (QuantConnect Research) =====
# Downloads continuous front-month 1-minute bars (all sessions, real prices) for several markets,
# 2022 -> today, and saves one gzipped CSV per market into this project's folder (shows up in Explorer).
from AlgorithmImports import *
import pandas as pd, numpy as np, os, gzip

MARKETS = ["CL", "GC", "SI", "HG", "NG", "NQ"]   # crude, gold, silver, copper, nat gas, nasdaq (comparison)
YEARS   = [2022, 2023, 2024, 2025, 2026]

qb = QuantBook()

# ---- find the project folder (where research.ipynb lives) so files appear in the Explorer
proj = None
for root in ('/', os.path.expanduser('~')):
    for dp, dn, fn in os.walk(root):
        if 'research.ipynb' in fn and 'main.py' in fn:
            proj = dp; break
        if dp.count(os.sep) > 6: dn[:] = []
    if proj: break
proj = proj or os.getcwd()
print('saving into:', proj)

for tk in MARKETS:
    try:
        fut = qb.add_future(tk, Resolution.MINUTE,
                            extended_market_hours=True,                          # keep every session; we filter later
                            data_normalization_mode=DataNormalizationMode.RAW,   # real traded prices
                            data_mapping_mode=DataMappingMode.OPEN_INTEREST,     # most active contract
                            contract_depth_offset=0)
    except Exception as ex:
        print(tk, 'could not be added:', ex); continue
    parts = []
    for year in YEARS:
        try:
            hist = qb.history(fut.symbol, datetime(year, 1, 1), datetime(year + 1, 1, 1), Resolution.MINUTE)
        except Exception as ex:
            print(f'  {tk} {year}: error {ex}'); continue
        if hist is None or len(hist) == 0:
            print(f'  {tk} {year}: no data'); continue
        df = hist.reset_index()[['time', 'open', 'high', 'low', 'close', 'volume']].set_index('time').sort_index()
        df = df[~df.index.duplicated(keep='last')]
        parts.append(df)
        print(f'  {tk} {year}: {len(df):,} bars')
    if not parts:
        continue
    df = pd.concat(parts).sort_index()
    df = df[~df.index.duplicated(keep='last')]
    if df.index.tz is None:
        df.index = df.index.tz_localize('America/New_York')
    out = os.path.join(proj, f'{tk}_1m_2022-2026.csv.gz')
    df.to_csv(out, compression='gzip')
    vol_by_hour = df.groupby(df.index.hour)['volume'].mean()
    print(f'{tk}: {len(df):,} bars | {df.index.min()} -> {df.index.max()} | busiest hour (ET) {vol_by_hour.idxmax()}:00 '
          f'| last close {df.close.iloc[-1]} | {os.path.getsize(out)/1e6:.1f} MB -> {os.path.basename(out)}')

print('done - refresh the Explorer, then right-click each .csv.gz -> Download')
